# langnostic — Colab runner (T4)

All model work happens here: dataset streaming, whisper transcription, LoRA training.
Only `results/cache/*.jsonl` (+ `.meta.json`) goes back to the laptop.

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available())

In [ ]:
# fresh session: clone; existing repo: pull
import os
REPO = 'https://github.com/Coding-With-Josh/langnostic.git'
if not os.path.isdir('/content/langnostic'):
    !git clone $REPO /content/langnostic
%cd /content/langnostic
!git pull --ff-only

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
# Build speaker-disjoint manifests: about 2.6 GB of pinned parquet, downloaded
# then decoded. One NaijaVoices shard supplies the training budget; VoxPopuli
# en, pooled L2-ARCTIC and FLEURS yo_ng are test-only.
!PYTHONPATH=. python -m speechgap.data build --out-dir data \
    --naija yoruba:0-0 \
    --voxpopuli en:test \
    --l2arctic \
    --fleurs yo_ng:test --allow-clip-level
!PYTHONPATH=. python -m speechgap.data verify --out-dir data

In [ ]:
import json
audit = json.load(open('data/audit.json'))
print(json.dumps({k: audit[k] for k in ('splits', 'groups', 'dropped', 'accents_seen', 'problems')}, indent=2))

In [ ]:
# Zero-shot baseline. Deterministic: pinned model revision, forced per-row language.
!PYTHONPATH=. python -m speechgap.eval --manifest data/manifest_test.jsonl \
    --device cuda --out results/cache/zero_shot.jsonl

# After a disconnect, rerun the same command with --resume.

In [ ]:
!PYTHONPATH=. python -m speechgap.report results/cache/zero_shot.jsonl \
    --out results/zero_shot_report.json

## LoRA: pilot first, then the budgeted runs

`speechgap.train` refuses to publish an adapter whose final validation output
fails the pre-registered gates (WER, wrong-script, degeneracy, forced language),
and prints three ref/hyp pairs to read. The pilot is cheap on purpose: if it
fails, the 15/30-minute budget is not worth spending.

In [ ]:
# Pilot: 2 minutes, one epoch, 20 validation clips. check=True means a failing
# gate raises here instead of letting the notebook continue to a 15/30-min run.
import os, subprocess, sys
os.environ['PYTHONPATH'] = '.'

def run(*args):
    print('$', ' '.join(args), flush=True)
    subprocess.run([sys.executable, '-m', *args], check=True)

run('speechgap.train', '--train', 'data/manifest_train.jsonl',
    '--val', 'data/manifest_val.jsonl', '--minutes', '2', '--epochs', '1',
    '--max-val', '20', '--patience', '1', '--out', 'checkpoints/yo-pilot',
    '--group', 'yoruba_read', '--language', 'yo')

In [ ]:
# Gated on the pilot: refuse to spend 15/30 minutes unless the pilot adapter
# exists and every pre-registered gate passed on its final validation output.
import json
meta = json.load(open('checkpoints/yo-pilot/train_meta.json', encoding='utf-8'))
gates = meta['pilot_gates']
assert gates['ok'], gates['failures']
print(json.dumps({'pilot_gates': gates, 'language_forced': meta['language_forced']},
                 indent=2, ensure_ascii=False))

for minutes in (15, 30):
    run('speechgap.train', '--train', 'data/manifest_train.jsonl',
        '--val', 'data/manifest_val.jsonl', '--minutes', str(minutes),
        '--group', 'yoruba_read', '--language', 'yo',
        '--out', f'checkpoints/yo-{minutes}min')

In [ ]:
!PYTHONPATH=. python -m speechgap.eval --manifest data/manifest_test.jsonl \
    --device cuda --adapter checkpoints/yo-15min --out results/cache/yo_lora15.jsonl
!PYTHONPATH=. python -m speechgap.eval --manifest data/manifest_test.jsonl \
    --device cuda --adapter checkpoints/yo-30min --out results/cache/yo_lora30.jsonl

for name in ('zero_shot', 'yo_lora15', 'yo_lora30'):
    !PYTHONPATH=. python -m speechgap.report results/cache/$name.jsonl --out results/${name}_report.json

In [ ]:
# Bring the cache home. Transcripts only: no audio, no manifests, no weights.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/langnostic/cache
!cp results/cache/*.jsonl results/cache/*.meta.json /content/drive/MyDrive/langnostic/cache/

## next

LoRA training is the remaining piece. It reads `data/manifest_train.jsonl` and
`data/manifest_val.jsonl`, trains speaker-disjoint from test, and writes adapters to
`checkpoints/` (never committed). Then evaluate every adapter on every group.